# Baseline Results

This notebook loads the benchmark empirical outputs behind the paper's main claim. By default it reads the generated CSV and PNG files under `output/`, but you can refresh the baseline analysis from here without changing the underlying pipeline architecture.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate the project root containing pyproject.toml.")
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

pd.options.display.max_columns = 50
pd.options.display.float_format = lambda value: f"{value:,.3f}"
plt.style.use("seaborn-v0_8-whitegrid")

from laos_fx_oil_macro.notebook_helpers import (
    build_notebook_project,
    compact_interval_pivot,
    load_csv,
    round_numeric_columns,
    run_pipeline_from_notebook,
)

project = build_notebook_project(PROJECT_ROOT)
project.root


In [ ]:
REFRESH_BASELINE = False
LP_HORIZONS = 12
LP_LAGS = 2
REFRESH_BASELINE


In [ ]:
if REFRESH_BASELINE:
    baseline_manifest = run_pipeline_from_notebook(
        project,
        lp_horizons=LP_HORIZONS,
        lp_lags=LP_LAGS,
        skip_tvp=True,
    )
    display(
        pd.DataFrame(
            {
                "artifact": baseline_manifest.keys(),
                "path": [str(path.relative_to(project.root)) for path in baseline_manifest.values()],
            }
        )
    )
else:
    print("REFRESH_BASELINE is False. Loading the current baseline outputs from disk.")


In [ ]:
required_paths = {
    "breaks": project.tables_dir / "break_diagnostics.csv",
    "regime_summary": project.tables_dir / "regime_summary.csv",
    "gpr_to_oil": project.tables_dir / "gpr_to_oil_hac_ols.csv",
    "lp_oil_to_fx": project.tables_dir / "lp_oil_to_fx.csv",
    "lp_oil_to_inflation": project.tables_dir / "lp_oil_to_inflation.csv",
    "lp_fx_to_inflation": project.tables_dir / "lp_fx_to_inflation.csv",
}
missing = [str(path.relative_to(project.root)) for path in required_paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing baseline outputs: " + ", ".join(missing))

breaks = load_csv(required_paths["breaks"])
regime_summary = load_csv(required_paths["regime_summary"])
gpr_to_oil = load_csv(required_paths["gpr_to_oil"])
lp_oil_to_fx = load_csv(required_paths["lp_oil_to_fx"])
lp_oil_to_inflation = load_csv(required_paths["lp_oil_to_inflation"])
lp_fx_to_inflation = load_csv(required_paths["lp_fx_to_inflation"])


## Break diagnostics


In [ ]:
display(round_numeric_columns(breaks))


The macro break is strongest in depreciation and inflation rather than in the mean oil shock itself. That is consistent with the paper's framing: the post-2022 Laos episode looks more like a transmission-regime change than a simple rise in average external oil shocks.


## Regime summary


In [ ]:
regime_means = regime_summary.pivot(index="variable", columns="regime", values="mean").reset_index()
display(round_numeric_columns(regime_means))


The regime averages show the same pattern in levels: post-2022 Laos has materially higher depreciation and inflation, while the oil shock series itself does not move to a comparably higher mean. This is why the paper emphasizes exchange-rate amplification under stress.


## HAC OLS: `GPR_ME -> oil_shock`


In [ ]:
gpr_display = gpr_to_oil[gpr_to_oil["term"] != "const"].copy()
gpr_display["estimate"] = gpr_display.apply(
    lambda row: f"{row['coefficient']:.3f} (p={row['p_value']:.3f})", axis=1
)
display(gpr_display.pivot(index="term", columns="regime", values="estimate"))


The direct monthly reduced-form `GPR_ME -> oil_shock` link is weak in both regimes. That matters for interpretation: the cleanest empirical result in this project is not a strong monthly geopolitical-risk estimate, but the stronger post-2022 `oil -> FX -> inflation` transmission pattern.


## Local projections: `oil_shock -> fx_dep`


In [ ]:
display(compact_interval_pivot(lp_oil_to_fx, index="horizon", columns="regime"))
figure_path = project.figures_dir / "lp_oil_to_fx.png"
if figure_path.exists():
    display(Image(filename=str(figure_path), width=760))


Before 2022, the oil-to-FX response is near zero. After the January 2022 break it turns positive at short horizons, which fits the idea that oil shocks became more likely to intensify dollar demand and kip weakness once macro-financial stress deepened.


## Local projections: `oil_shock -> inflation_mom`


In [ ]:
display(compact_interval_pivot(lp_oil_to_inflation, index="horizon", columns="regime"))
figure_path = project.figures_dir / "lp_oil_to_inflation.png"
if figure_path.exists():
    display(Image(filename=str(figure_path), width=760))


The oil-to-inflation response is positive in both subsamples, but the post-2022 pattern is more delayed and peaks later. That is consistent with external price pressure working through a stressed exchange-rate and balance-sheet environment rather than only through immediate fuel-price pass-through.


## Local projections: `fx_dep -> inflation_mom`


In [ ]:
display(compact_interval_pivot(lp_fx_to_inflation, index="horizon", columns="regime"))
figure_path = project.figures_dir / "lp_fx_to_inflation.png"
if figure_path.exists():
    display(Image(filename=str(figure_path), width=760))

fx_h0 = lp_fx_to_inflation.loc[lp_fx_to_inflation["horizon"] == 0, ["regime", "coefficient", "lower_90", "upper_90", "nobs"]]
display(round_numeric_columns(fx_h0))


This is the key paper result. The contemporaneous post-2022 pass-through from depreciation to monthly inflation is much stronger than in the pre-2022 sample, rising from about `0.04` before the break to about `0.49` after it. That gap is the cleanest empirical support for the paper's claim that debt distress amplified exchange-rate pass-through in Laos.
